In [5]:
import pandas as pd

# Загрузка данных из CSV-файла
data = pd.read_csv('data/weatherAUS.csv', na_values='NA')

# Подсчет общего количества пропусков в данных
total_missing = data.isna().sum().sum()

print("Суммарное количество пропусков в данных:", total_missing)

Суммарное количество пропусков в данных: 343248


In [6]:
# Общее количество строк в данных
total_rows = len(data)

# Подсчет процента пропусков для каждого столбца
missing_percentages = data.isna().mean() * 100

# Определение столбцов, где пропусков более 40%
columns_to_drop = missing_percentages[missing_percentages > 40].index

# Количество столбцов с пропусками более 40%
num_columns_dropped = len(columns_to_drop)

# Удаление этих столбцов из DataFrame
data = data.drop(columns=columns_to_drop)

# Вывод количества удаленных столбцов
print("Количество признаков с пропусками более 40%:", num_columns_dropped)
print("Удаленные признаки:", list(columns_to_drop))

Количество признаков с пропусками более 40%: 3
Удаленные признаки: ['Evaporation', 'Sunshine', 'Cloud3pm']


In [7]:
# Замена значений в столбцах RainToday и RainTomorrow, сохраняя пропуски
data['RainToday'] = data['RainToday'].replace({'Yes': 1, 'No': 0})
data['RainTomorrow'] = data['RainTomorrow'].replace({'Yes': 1, 'No': 0})

# Вычисление среднего арифметического для преобразованного RainToday
mean_rain_today = data['RainToday'].mean()

# Округление до двух знаков после точки
mean_rain_today_rounded = round(mean_rain_today, 2)

print("Среднее арифметическое для преобразованного признака RainToday:", mean_rain_today_rounded)

Среднее арифметическое для преобразованного признака RainToday: 0.22


In [8]:
# Преобразование столбца Date в формат datetime
data['Date'] = pd.to_datetime(data['Date'])

# Выделение номера месяца в новый столбец Month
data['Month'] = data['Date'].dt.month

# Удаление столбца Date
data = data.drop(columns=['Date'])

# Группировка по месяцу и вычисление доли дождливых дней (RainToday == 1)
rainy_days_by_month = data.groupby('Month')['RainToday'].mean()

# Нахождение месяца с максимальной долей дождливых дней
max_rainy_month = rainy_days_by_month.idxmax()

print("Месяц с наибольшей долей дождливых дней:", max_rainy_month)

Месяц с наибольшей долей дождливых дней: 7


In [9]:
# Список категориальных признаков
categoricals = ['Month', 'Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']

# Применение get_dummies для создания dummy-переменных
data = pd.get_dummies(data, columns=categoricals)

# Подсчет общего количества признаков, включая целевую переменную RainTomorrow
num_features = data.shape[1]

print("Общее количество признаков в данных, включая целевую переменную:", num_features)

Общее количество признаков в данных, включая целевую переменную: 124


In [10]:
from sklearn.model_selection import train_test_split

# Удаление строк с пропусками
data = data.dropna()

# Разделение данных на признаки (X) и целевую переменную (y)
X = data.drop(columns=['RainTomorrow'])
y = data['RainTomorrow']

# Разбиение на обучающую и тестовую выборки (70/30, random_state=31)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=31)

# Вычисление среднего значения целевой переменной на тестовой выборке
mean_rain_tomorrow = y_test.mean()

# Округление до двух знаков после точки
mean_rain_tomorrow_rounded = round(mean_rain_tomorrow, 2)

print("Среднее значение целевой переменной на тестовой выборке:", mean_rain_tomorrow_rounded)

Среднее значение целевой переменной на тестовой выборке: 0.23


In [11]:
import numpy as np

# Фиксация случайности
np.random.seed(31)

# Размер обучающей выборки
n_train = len(X_train)

# Количество бутстреп-выборок
n_bootstrap = 1000

# Список для хранения средних значений MinTemp по каждой бутстреп-выборке
bootstrap_means = []

# Генерация 1000 бутстреп-выборок
for _ in range(n_bootstrap):
    # Генерация случайных индексов с возвращением
    indices = np.random.randint(0, n_train, size=n_train)
    # Извлечение выборки по индексам
    bootstrap_sample = X_train['MinTemp'].iloc[indices]
    # Вычисление среднего значения MinTemp для выборки
    bootstrap_means.append(bootstrap_sample.mean())

# Вычисление стандартного отклонения средних значений
std_bootstrap_means = np.std(bootstrap_means)

# Округление до двух знаков после точки
std_bootstrap_means_rounded = round(std_bootstrap_means, 2)

print("Стандартное отклонение среднего значения MinTemp:", std_bootstrap_means_rounded)

Стандартное отклонение среднего значения MinTemp: 0.03


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Создание и обучение модели логистической регрессии
model = LogisticRegression()
model.fit(X_train, y_train)

# Предсказание вероятностей на тестовой выборке
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Вычисление ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Округление до двух знаков после точки
roc_auc_rounded = round(roc_auc, 2)

print("ROC-AUC на тестовой выборке:", roc_auc_rounded)

ROC-AUC на тестовой выборке: 0.87


C:\Users\mi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

# Определение сетки гиперпараметров
params = {
    'max_leaf_nodes': list(range(2, 10)),
    'min_samples_split': [2, 3, 4],
    'max_depth': [5, 7, 9, 11]
}

# Создание модели решающего дерева
dt = DecisionTreeClassifier(random_state=42)

# Настройка GridSearchCV
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=params,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

# Обучение модели с перебором гиперпараметров
grid_search.fit(X_train, y_train)

# Получение лучшей модели
best_model = grid_search.best_estimator_

# Предсказание вероятностей на тестовой выборке
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Вычисление ROC-AUC для лучшей модели
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Округление до двух знаков после точки
roc_auc_rounded = round(roc_auc, 2)

# Получение оптимальных гиперпараметров
best_params = grid_search.best_params_

print("ROC-AUC на тестовой выборке для оптимального дерева решений:", roc_auc_rounded)
print("Оптимальные гиперпараметры:")
print("max_depth:", best_params['max_depth'])
print("max_leaf_nodes:", best_params['max_leaf_nodes'])
print("min_samples_split:", best_params['min_samples_split'])

ROC-AUC на тестовой выборке для оптимального дерева решений: 0.82
Оптимальные гиперпараметры:
max_depth: 5
max_leaf_nodes: 9
min_samples_split: 2


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Создание и обучение модели случайного леса
rf = RandomForestClassifier(n_estimators=100, random_state=31)
rf.fit(X_train, y_train)

# Предсказание вероятностей на тестовой выборке
y_pred_proba = rf.predict_proba(X_test)[:, 1]

# Вычисление ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Округление до двух знаков после точки
roc_auc_rounded = round(roc_auc, 2)

print("ROC-AUC на тестовой выборке для случайного леса:", roc_auc_rounded)

ROC-AUC на тестовой выборке для случайного леса: 0.89


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

# Определение сетки гиперпараметров
params = {
    'max_features': [4, 5, 6, 7],
    'min_samples_leaf': [3, 5, 7, 9, 11],
    'max_depth': [5, 10, 15]
}

# Создание модели случайного леса
rf = RandomForestClassifier(n_estimators=100, random_state=31)

# Настройка GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=params,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1
)

# Обучение модели с перебором гиперпараметров
grid_search.fit(X_train, y_train)

# Получение лучшей модели
best_model = grid_search.best_estimator_

# Предсказание вероятностей на тестовой выборке
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Вычисление ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Округление до двух знаков после точки
roc_auc_rounded = round(roc_auc, 2)

print("ROC-AUC на тестовой выборке для оптимального случайного леса:", roc_auc_rounded)

ROC-AUC на тестовой выборке для оптимального случайного леса: 0.88


In [16]:
import pandas as pd

# Получение важности признаков из лучшей модели
feature_importances = best_model.feature_importances_

# Создание DataFrame с названиями признаков и их важностью
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
})

# Сортировка по убыванию важности
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Вывод трех признаков с наибольшей важностью
top_3_features = importance_df.head(3)

print("Три признака с наибольшим вкладом в целевую переменную:")
print(top_3_features)

Три признака с наибольшим вкладом в целевую переменную:
       Feature  Importance
7  Humidity3pm    0.250783
2     Rainfall    0.079757
6  Humidity9am    0.070403
